# Loading packages

In [1]:
from plotnine import *
import pandas as pd
from Bio.SeqUtils import gc_fraction
import pysam
from pathlib import Path
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import os
import pickle
from tqdm.autonotebook import tqdm
import numpy as np

/tmp/ipykernel_826103/176058627.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)


In [2]:
%matplotlib inline

# Definition of functions to read in files

In [ ]:
def get_files(
    sample,
    genus,
    fasta_basepath = "<path to>/Pseudobins_Core_Genera/fasta",
    pydam_basepath = "<path to>/Damage_Inference_Core_Genera/results/pydamage",
    md2_basepath = "<path to>/Damage_Inference_Core_Genera/results/mapdamage",
    length_basepath = "<path to>/Damage_Inference_Core_Genera/results/compute_length"
):
    fasta_file = os.path.join(fasta_basepath, f"{sample}_{genus}.fa")
    pydam_file = os.path.join(pydam_basepath, f"{sample}-{genus}", "pydamage_results", f"{sample}_pydamage_results.csv")
    mapdam_file = os.path.join(md2_basepath, f"{sample}-{genus}", f"results_{sample}_cleaned", "Stats_out_MCMC_iter_summ_stat.csv")
    length_file = os.path.join(length_basepath, f"{sample}-{genus}", f"{sample}_params.tsv")



    return fasta_file, pydam_file, mapdam_file, length_file


In [4]:
def read_mapdamage(f):
    s = pd.read_csv(f, sep=",", header=0, index_col=0).loc["Mean",:]
    return {
        'tool'    : "mapdamage",
        'delta_d' : s["DeltaD"],
        'delta_s' : s["DeltaS"],
        'lambda'  : s["Lambda"], 
    }

In [5]:
def read_pydamage(f):
    d = pd.read_csv(f, sep=",", header=0, index_col=0)
    return {
        'tool'    : "pydamage",
        'pmin' : d['damage_model_pmin'].mean(),
        'pmin_SD' : d['damage_model_pmin'].std(),
        'pmax' : d["damage_model_pmax"].mean(),
        'pmax_SD': d["damage_model_pmax"].std(),
        'p'  : d["damage_model_p"].mean(),
        'p_SD': d["damage_model_p"].std(),
        'nb_contigs' : len(d),
        'nb_reads' : d["nb_reads_aligned"].sum(),
        'len' : d["reflen"].sum()
    }
    return s

In [6]:
def read_length(f):
    d = pd.read_csv(f, sep="\t", header=0)
    return {
        'tool'    : "compute_length",
        'mode'    : d["mode_geom"][0],
        "geom_p"  : d["p"][0],
    }


In [7]:
def compute_gc(fasta):
    gc = []
    with pysam.FastxFile(fasta) as fa:
        for contig in fa:
            gc.append(gc_fraction(contig.sequence))
    return {
        'gc' : np.mean(gc),
    }

# Read the data

In [ ]:
genus_df = pd.read_csv("../01_nf-damage-inference/01_core_genera/genomes.csv")

In [9]:
samples = genus_df["sample"].unique()
genus = genus_df["taxon"].unique()

In [10]:
res_list = []
for s in samples:
    for g in genus:
        fa, pydam, md2, l = get_files(s, g)
        if os.path.exists(fa):
            gc_d = compute_gc(fa)
            gc_d.update({"tool": "gc", "sample": s, "genus": g})
            res_list.append(gc_d)
        if os.path.exists(pydam):
            pyd_d = read_pydamage(pydam)
            pyd_d.update({"sample": s, "genus": g})
            res_list.append(pyd_d)
        if os.path.exists(md2):
            md2_d = read_mapdamage(md2)
            md2_d.update({"sample": s, "genus": g})
            res_list.append(md2_d)
        if os.path.exists(l):
            l_d = read_length(l)
            l_d.update({"sample": s, "genus": g})
            res_list.append(l_d)

In [11]:
nf_dam_inf_res = pd.DataFrame(res_list).melt(id_vars=["sample", "genus", "tool"]).dropna(subset=["value"])
nf_dam_inf_res


,sample,genus,tool,variable,value
0,TDM033,nitrospira,gc,gc,0.576604
4,TDM033,conexibacter,gc,gc,0.693642
8,TDM033,piscinibacter,gc,gc,0.701847
12,TDM033,candidatus_nitrosotenuis,gc,gc,0.471471
16,TDM033,nitrosotalea,gc,gc,0.433031
...,...,...,...,...,...
20473,TDM058,nitrospira,compute_length,geom_p,0.083537
20477,TDM058,conexibacter,compute_length,geom_p,0.095870
20481,TDM058,piscinibacter,compute_length,geom_p,0.048317
20485,TDM058,candidatus_nitrosotenuis,compute_length,geom_p,0.062494


In [ ]:
age_model = pd.read_csv("<path to>/summarized_age_model.csv", sep=' ', decimal='.')
age_model['sample'] = age_model['sample_name']
age_model

,sample_name,dft_interval,min_age_yrs_bf_1950,max_age_yrs_bf_1950,mean_age_yrs_bf_1950,median_age_yrs_bf_1950,ImageJ_measurement_cm,Rounded_Length,DFT,Rounded_DFT,sample
0,TDM001,"(0,1]",80,615,347.350000,347.0,0.93,1.00,1.000,1,TDM001
1,TDM002,"(1,2]",644,1186,917.000000,917.5,0.95,1.00,2.000,2,TDM002
2,TDM003,"(2,3]",1214,1716,1468.000000,1469.5,1.00,1.00,3.000,3,TDM003
3,TDM004,"(3,4]",1742,2191,1972.350000,1976.0,0.98,1.00,4.000,4,TDM004
4,TDM005,"(4,5]",2214,2670,2439.250000,2438.0,0.88,1.00,5.000,5,TDM005
...,...,...,...,...,...,...,...,...,...,...,...
56,TDM057,"(55.2,56.2]",32725,33613,33170.150000,33171.5,1.00,1.00,56.250,57,TDM057
57,TDM058,"(56.2,57.2]",33660,34540,34100.400000,34101.0,0.95,1.00,57.250,58,TDM058
58,TDM059,"(57.2,58.2]",34586,35465,35024.800000,35025.0,1.11,1.00,58.250,59,TDM059
59,TDM060,"(58.2,59.5]",35512,36626,36069.160000,36070.0,1.22,1.25,59.500,60,TDM060


In [20]:
nf_dam_inf_res = nf_dam_inf_res.merge(age_model, on="sample")
nf_dam_inf_res[nf_dam_inf_res['value'].isna()]

,sample,genus,tool,variable,value,sample_name,dft_interval,min_age_yrs_bf_1950,max_age_yrs_bf_1950,mean_age_yrs_bf_1950,median_age_yrs_bf_1950,ImageJ_measurement_cm,Rounded_Length,DFT,Rounded_DFT


In [ ]:
labdata=pd.read_csv("<path to>/lab_parameters.txt", sep="\t", decimal='.').rename(columns={"Pandora_ID": "sample"})
labdata

,Library,qPCRcopies_per_uL,sample,mass_mg,EDTA_fractions
0,TDM001.A0201,57200000.0,TDM001,475.5,1
1,TDM002.A0201,246000000.0,TDM002,830.9,1
2,TDM003.A0201,62000000.0,TDM003,808.5,1
3,TDM004.A0201,96300000.0,TDM004,731.3,1
4,TDM005.A0201,108000000.0,TDM005,699.0,1
...,...,...,...,...,...
56,TDM057.A0301,15800000.0,TDM057,3589.7,2
57,TDM058.A0301,12600000.0,TDM058,3635.8,2
58,TDM059.A0301,10100000.0,TDM059,4035.8,2
59,TDM060.A0301,16900000.0,TDM060,4191.9,2


In [22]:
nf_dam_inf_res = nf_dam_inf_res.merge(labdata, on="sample")
nf_dam_inf_res[nf_dam_inf_res['value'].isna()]

,sample,genus,tool,variable,value,sample_name,dft_interval,min_age_yrs_bf_1950,max_age_yrs_bf_1950,mean_age_yrs_bf_1950,median_age_yrs_bf_1950,ImageJ_measurement_cm,Rounded_Length,DFT,Rounded_DFT,Library,qPCRcopies_per_uL,mass_mg,EDTA_fractions


In [23]:
nf_dam_inf_res['tool_param'] = nf_dam_inf_res['tool'] + "_" + nf_dam_inf_res['variable']
len(list(set(nf_dam_inf_res['sample'])))
nf_dam_inf_res['tool_param'].value_counts()

tool_param
gc_gc                    341
pydamage_pmin            341
pydamage_pmax            341
pydamage_p               341
pydamage_nb_contigs      341
pydamage_nb_reads        341
pydamage_len             341
mapdamage_delta_d        341
mapdamage_delta_s        341
mapdamage_lambda         341
compute_length_mode      341
compute_length_geom_p    341
pydamage_pmin_SD         306
pydamage_pmax_SD         306
pydamage_p_SD            306
Name: count, dtype: int64

In [ ]:
nf_dam_inf_res.to_csv('damage_data_core_genera.csv', sep=',', index=False, encoding='utf-8')
